In [2]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as pyplot 
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

In [3]:
df = pd.read_csv('titanic_toy.csv')

In [4]:
df.sample(5)

,Age,Fare,Family,Survived
515,47.0,34.0208,0,0
630,80.0,30.0000,0,1
516,34.0,10.5000,0,1
839,NaN,29.7000,0,1
856,45.0,164.8667,2,1


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Age       714 non-null    float64
 1   Fare      846 non-null    float64
 2   Family    891 non-null    int64  
 3   Survived  891 non-null    int64  
dtypes: float64(2), int64(2)
memory usage: 28.0 KB


In [6]:
df.isnull().mean()

Age         0.198653
Fare        0.050505
Family      0.000000
Survived    0.000000
dtype: float64

In [7]:
x = df.drop(columns=['Survived'])
y = df['Survived']

In [8]:
xtrain , xtest, ytrain, ytest = train_test_split(x,y, test_size=0.2,random_state=42)

In [9]:
xtrain.shape , ytrain.shape

((712, 3), (712,))

In [10]:
mean_age = xtrain['Age'].mean()
med_age = xtrain['Age'].median()

mean_fare = xtrain['Fare'].mean()
med_fare = xtrain['Fare'].median()

In [11]:
xtrain['Age mean'] = xtrain['Age'].fillna(mean_age)
xtrain['Age Med'] = xtrain['Age'].fillna(med_age)

xtrain['Fare mean'] = xtrain['Fare'].fillna(mean_fare)
xtrain['Fare Med'] = xtrain['Fare'].fillna(med_fare)

In [12]:
xtrain.sample(5)

,Age,Fare,Family,Age mean,Age Med,Fare mean,Fare Med
879,56.0,83.1583,1,56.0,56.0,83.1583,83.1583
498,25.0,151.5500,3,25.0,25.0,151.5500,151.5500
170,61.0,33.5000,0,61.0,61.0,33.5000,33.5000
138,16.0,9.2167,0,16.0,16.0,9.2167,9.2167
58,5.0,27.7500,3,5.0,5.0,27.7500,27.7500


In [13]:
print('Original Age variable variance: ', xtrain['Age'].var())
print('Age Variance after median imputation: ', xtrain['Age Med'].var())
print('Age Variance after mean imputation: ', xtrain['Age mean'].var())

print('Original Fare variable variance: ', xtrain['Fare'].var())
print('Fare Variance after median imputation: ', xtrain['Fare Med'].var())
print('Fare Variance after mean imputation: ', xtrain['Fare mean'].var())

Original Age variable variance:  210.2517072477438
Age Variance after median imputation:  169.20731007048096
Age Variance after mean imputation:  168.8519336687225
Original Fare variable variance:  2761.031434948639
Fare Variance after median imputation:  2637.01248167777
Fare Variance after mean imputation:  2621.2323749512393


In [14]:
xtrain.cov()

,Age,Fare,Family,Age mean,Age Med,Fare mean,Fare Med
Age,210.251707,75.481375,-6.993325,210.251707,210.251707,71.193767,70.082085
Fare,75.481375,2761.031435,18.599163,60.224654,63.938058,2761.031435,2761.031435
Family,-6.993325,18.599163,2.830892,-5.616299,-5.587710,17.657433,17.672035
Age mean,210.251707,60.224654,-5.616299,168.851934,168.851934,57.175304,56.282518
Age Med,210.251707,63.938058,-5.587710,168.851934,169.207310,60.700688,59.728510
Fare mean,71.193767,2761.031435,17.657433,57.175304,60.700688,2621.232375,2621.232375
Fare Med,70.082085,2761.031435,17.672035,56.282518,59.728510,2621.232375,2637.012482


### Using sklearn

In [17]:
xtrain , xtest , ytrain,ytest = train_test_split(x,y,train_size=0.2 , random_state=42)

In [19]:
imputer1 = SimpleImputer(strategy='median')
imputer2  = SimpleImputer(strategy='mean')

In [21]:
trf = ColumnTransformer([('imputer1',imputer1,['Age']),
                        ('imputer2',imputer2,['Fare'])],remainder='passthrough')

In [23]:
trf.fit(xtrain)

C:\Users\dkglt\anaconda3\Lib\site-packages\sklearn\compose\_column_transformer.py:1623: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(


ColumnTransformer(remainder='passthrough',
                  transformers=[('imputer1', SimpleImputer(strategy='median'),
                                 ['Age']),
                                ('imputer2', SimpleImputer(), ['Fare'])])

In [27]:
trf.named_transformers_['imputer1'].statistics_


array([30.])

In [29]:
trf.named_transformers_['imputer2'].statistics_


array([28.5883872])

In [31]:
xtrain = trf.transform(xtrain)
xtest = trf.transform(xtest)

In [33]:
xtrain

array([[ 41.       ,   7.125    ,   0.       ],
       [ 48.       ,  76.7292   ,   1.       ],
       [ 48.       ,  65.       ,   3.       ],
       [ 48.       ,  39.6      ,   1.       ],
       [  4.       ,  31.275    ,   6.       ],
       [ 39.       ,  55.9      ,   1.       ],
       [ 33.       ,  15.85     ,   3.       ],
       [ 29.       ,  10.5      ,   0.       ],
       [ 49.       ,  28.5883872,   2.       ],
       [ 30.       ,  69.55     ,  10.       ],
       [ 30.       ,   7.2292   ,   0.       ],
       [ 42.       ,  52.       ,   1.       ],
       [ 36.       ,  71.       ,   2.       ],
       [ 61.       ,  33.5      ,   0.       ],
       [ 18.       ,   6.75     ,   0.       ],
       [ 18.       ,   7.4958   ,   0.       ],
       [  0.67     ,  14.5      ,   2.       ],
       [ 32.       ,  56.4958   ,   0.       ],
       [ 30.       ,   8.05     ,   0.       ],
       [ 30.       ,   8.05     ,   0.       ],
       [ 30.       ,  14.4542   ,   1.  